In [1]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    markers = ["README.md", ".git", "Models", "utils.py"]
    for d in [p] + list(p.parents):
        if any((d / m).exists() for m in markers):
            return d
    return p

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [2]:
from functools import partial
from utils import *
from Models.Generative_Closures.Generative_Models import *
from Models.Generative_Closures.Interpolant import *
from Models.Pretrained_Autoencoders.AE import *

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
plt.rc("text", usetex=True)
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["text.latex.preamble"] = r"\usepackage{amsmath}"

In [3]:
# Check if CUDA is available
if torch.cuda.is_available():
    print("CUDA is available.")
    device = torch.device('cuda')
else:
    print("CUDA is not available.")
    device = torch.device('cpu')

# Load the data
DATA_PATH = ROOT / "Data" / "test_diffusion_nonlinear_B100.h5"
N = 10
with h5py.File(DATA_PATH, 'r') as file:
    test_nonlinear = torch.tensor(file['test_nonlinear_64'][:N], device=device)
    test_vorticity = torch.tensor(file['test_vorticity_64'][:N], device=device)

metrics = ErrorMetrics()

CUDA is available.


In [4]:
def sample_closure_field(
    method: str,
    model: torch.nn.Module,
    omega: torch.Tensor,
    num_ensembles: int = 1,
    sample_steps: int = 100,
    sigma_coef: float = 1.0,
    time_min: float = 1e-3,
    time_max: float = 1.0,
    time_schedule: str = 'uniform',   # 'uniform' or 'karras'
    return_starting_field: bool = False,
    device: str = 'cuda',
):
    """
    Unified closure sampler for 'diffusion', 'onesided', and 'twosided' methods.
    - 'DM'  : standard score-based SDE sampler
    - 'FM'   : flow matching sampler
    - 'SI_known' : stochastic interpolant sampler with known base dist
    - 'SI_gaussian' : stochastic interpolant sampler with gaussian base dist

    Returns:
        Tensor of shape (B, num_ensembles, H, W).  If return_starting_field=True,
    """
    B, H, W = omega.shape
    omega = omega.to(device)
    model = model.to(device)
    model.eval()

    # 1) Build the time grid ts = [time_max, ..., time_min]
    if time_schedule == 'uniform':
        ts = torch.linspace(time_max, time_min, sample_steps+1, device=device)
        ts_forward = torch.linspace(time_min, time_max, sample_steps+1, device=device)
    elif time_schedule == 'karras':
        ts = get_sigmas_karras(sample_steps, time_min, time_max, device=device)
        ts_forward = reversed(ts)
    else:
        raise ValueError(f"Unknown time_schedule: {time_schedule}")
    dt = ts[:-1] - ts[1:]  # step sizes for Euler integration
    dt_forward = ts_forward[1:] - ts_forward[:-1]

    # 2) Tile conditioning omega to match ensemble dimension
    omega_rep = omega.unsqueeze(1).repeat(1, num_ensembles, 1, 1).view(-1, H, W)

    if method == 'DM':
        marginal_prob_std_fn = partial(marginal_prob_std, sigma=sigma_coef, device_=device)
        diffusion_coeff_fn = partial(diffusion_coeff, sigma=sigma_coef, device_=device)
        # 3) Initialize x at t = ts[0] ~ N(0, [marginal std]^2)
        t0 = ts[0].repeat(B * num_ensembles)
        x = torch.randn_like(omega_rep) * marginal_prob_std_fn(t0)[:, None, None]
        x0 = x.clone()  # store initial noise if requested

        total_distance = torch.zeros(B * num_ensembles, device=device)
        x_prev = x.clone()


        with torch.no_grad():
            for i in range(sample_steps):
                t_i = ts[i].repeat(B * num_ensembles)
                step_size = dt[i]

                g = diffusion_coeff_fn(t_i)        # shape [B*num_ensembles]
                grad = model(t_i, x, omega_rep)    # score: [B*num_ensembles, H, W]

                # Euler–Maruyama update:
                mean = x + (g**2)[:, None, None] * grad * step_size
                if i < sample_steps - 1:
                    noise = torch.randn_like(x)
                    x = mean + math.sqrt(step_size) * g[:, None, None] * noise
                else:
                    x = mean # no extra noise on final step

                dist = torch.sqrt(torch.sum((x - x_prev)**2, dim=(1,2)))
                total_distance += dist

                # prepare for next iteration
                x_prev.copy_(x)

        x = x.view(B, num_ensembles, H, W)
        total_distance = total_distance.view(B, num_ensembles)
        if return_starting_field:
            x0 = x0.view(B, num_ensembles, H, W)
            return x0, x, total_distance
        else:
            return x

    elif method == 'SI_known':
        # 3) Initialize X_t at t=ts[0]=1 as x1 = omega_rep
        Xt = omega_rep.clone()
        X0 = Xt.clone()  # store if requested


        total_distance = torch.zeros(B * num_ensembles, device=device)
        x_prev = Xt.clone()

        with torch.no_grad():
            for i in range(sample_steps):
                t = ts_forward[i]                                  # scalar in [0,1]
                step_size = dt_forward[i]                          # > 0
                t_batch = t.repeat(B * num_ensembles).to(device)
                drift = model(t_batch, Xt, omega_rep)      # [B*num_ensembles, H, W]
                gdot_scalar = 1 - t
                gdot = gdot_scalar.repeat(B * num_ensembles)
                gdot = gdot.view([B * num_ensembles] + [1] * (Xt.ndim - 1))

                # 4.3) Euler–Maruyama backward step (no sign flip: dt>0, but X_{t-dt}=X_t - drift*dt + ...)
                if i < sample_steps - 1:
                    noise = torch.randn_like(Xt)
                    # Xt = Xt + drift * step_size
                    Xt = Xt + drift * step_size + gdot * math.sqrt(step_size) * noise
                else:
                    Xt = Xt + drift * step_size

                dist = torch.sqrt(torch.sum((Xt - x_prev)**2, dim=(1,2)))
                total_distance += dist

                # prepare for next iteration
                x_prev.copy_(Xt)

        Xt = Xt.view(B, num_ensembles, H, W)
        total_distance = total_distance.view(B, num_ensembles)
        if return_starting_field:
            X0 = X0.view(B, num_ensembles, H, W)
            return X0, Xt, total_distance
        else:
            return Xt


    elif method == 'SI_gaussian':
        # 3) Initialize X_t at t=ts[0]=1 as x1 = omega_rep
        Xt = torch.randn_like(omega_rep)
        X0 = Xt.clone()  # store if requested

        total_distance = torch.zeros(B * num_ensembles, device=device)
        x_prev = Xt.clone()

        with torch.no_grad():
            for i in range(sample_steps):
                t = ts_forward[i]                                  # scalar in [0,1]
                step_size = dt_forward[i]                          # > 0
                t_batch = t.repeat(B * num_ensembles).to(device)
                drift = model(t_batch, Xt, omega_rep)      # [B*num_ensembles, H, W]
                gdot_scalar = 1 - t
                gdot = gdot_scalar.repeat(B * num_ensembles)
                gdot = gdot.view([B * num_ensembles] + [1] * (Xt.ndim - 1))

                # 4.3) Euler–Maruyama backward step (no sign flip: dt>0, but X_{t-dt}=X_t - drift*dt + ...)
                if i < sample_steps - 1:
                    noise = torch.randn_like(Xt)
                    Xt = Xt + drift * step_size + gdot * math.sqrt(step_size) * noise
                else:
                    Xt = Xt + drift * step_size

                dist = torch.sqrt(torch.sum((Xt - x_prev)**2, dim=(1,2)))
                total_distance += dist

                # prepare for next iteration
                x_prev.copy_(Xt)

        Xt = Xt.view(B, num_ensembles, H, W)
        total_distance = total_distance.view(B, num_ensembles)
        if return_starting_field:
            X0 = X0.view(B, num_ensembles, H, W)
            return X0, Xt, total_distance
        else:
            return Xt

    elif method == 'FM':
        # 3) Initialize X_t at t=ts[0]=1 as x1 = omega_rep
        Xt = torch.randn_like(omega_rep)
        X0 = Xt.clone()  # store if requested
        total_distance = torch.zeros(B * num_ensembles, device=device)
        x_prev = Xt.clone()
        with torch.no_grad():
            for i in range(sample_steps):
                t = ts_forward[i]                                  # scalar in [0,1]
                step_size = dt_forward[i]                          # > 0
                t_batch = t.repeat(B * num_ensembles).to(device)
                drift = model(t_batch, Xt, omega_rep)      # [B*num_ensembles, H, W]
                Xt = Xt + drift * step_size

                dist = torch.sqrt(torch.sum((Xt - x_prev)**2, dim=(1,2)))
                total_distance += dist

                # prepare for next iteration
                x_prev.copy_(Xt)
        Xt = Xt.view(B, num_ensembles, H, W)
        total_distance = total_distance.view(B, num_ensembles)
        if return_starting_field:
            X0 = X0.view(B, num_ensembles, H, W)
            return X0, Xt, total_distance
        else:
            return Xt

    else:
        raise ValueError(f"Unknown sampling method: {method}")


We compare four latent-space setups:

1. **Reconstruction-only AE** (no regularization)  
2. **Geometry-aware (GA) regularized AE**  
3. **Metric-preserving (MP) regularized AE**  
4. **Jointly trained latent spaces** (end-to-end with diffusion / flow matching / stochastic interpolants)

Performance can be evaluated by swapping in the corresponding autoencoders and latent generative models in the following code cells.

## Key takeaways
- **Reconstruction-only** yields the **lowest reconstruction error**, but the latent geometry can become **arbitrarily distorted**, making it **poorly suited for latent generative modeling**.  
- **GA/MP regularization** suppresses these distortions; latent generative models trained in such spaces **nearly match physical-space performance** while being **much faster** due to **lower-dimensional inference**.  
- When explicit regularizers are undesirable, **joint training** lets the generative objective **shape a usable latent space**, effectively avoiding distortion and **improving end-to-end generation accuracy**.

In [ ]:
nonlinear_AE = VariationalAutoEncoder().to(device)
vorticity_AE = VariationalAutoEncoder().to(device)
NONLINEAR_AE_PATH = ROOT / "Trained_Models" / "AE" / "Nonlinear" / "Joint_AE_Nonlinear_DM.pth"
VORTICITY_AE_PATH = ROOT / "Trained_Models" / "AE" / "Vorticity" / "Joint_AE_Vorticity_DM.pth"
nonlinear_AE.load_state_dict(torch.load(NONLINEAR_AE_PATH))
vorticity_AE.load_state_dict(torch.load(VORTICITY_AE_PATH))

<All keys matched successfully>

In [6]:
with torch.no_grad():
    latent_nonlinear = nonlinear_AE.encode(test_nonlinear)
    latent_vorticity = vorticity_AE.encode(test_vorticity)
    
    recon_nonlinear = nonlinear_AE.decode(latent_nonlinear)
    recon_vorticity = vorticity_AE.decode(latent_vorticity)

In [8]:
AE_recon_re_nonlinear = metrics.frobenius(recon_nonlinear, test_nonlinear)
AE_recon_re_vorticity = metrics.frobenius(recon_vorticity, test_vorticity)
AE_recon_mse_nonlinear = metrics.mse(recon_nonlinear, test_nonlinear)
AE_recon_mse_vorticity = metrics.mse(recon_vorticity, test_vorticity)

print(f"AE recon relative L2 error (nonlinear): {AE_recon_re_nonlinear:.4e}")
print(f"AE recon relative L2 error (vorticity): {AE_recon_re_vorticity:.4e}")
print(f"AE recon MSE (nonlinear): {AE_recon_mse_nonlinear:.4e}")
print(f"AE recon MSE (vorticity): {AE_recon_mse_vorticity:.4e}")

AE recon relative L2 error (nonlinear): 4.0312e-02
AE recon relative L2 error (vorticity): 2.4938e-03
AE recon MSE (nonlinear): 1.0039e-04
AE recon MSE (vorticity): 6.2955e-06


Latent Space Diffusion Models

In [ ]:
set_seed(42)
num_ensembles = 100
sigma = 30
marginal_prob_std_fn = partial(marginal_prob_std, sigma=sigma, device_=device)

modes = 4
width = 20
Latent_Diffusion = FNO2d_Diffusion(marginal_prob_std_fn, modes, modes, width, padding = 0, embed_dim = 256, length = 1).to(device)

model_name = ROOT / "Trained_Models" / "DM" / "Latent_DM" / "Joint_DM.pth"
Latent_Diffusion.load_state_dict(torch.load(model_name))

latent_starting_fields, latent_samples, latent_distances = sample_closure_field(
    method='DM',
    model=Latent_Diffusion,
    omega=latent_vorticity.float(),
    num_ensembles=num_ensembles,
    sample_steps=10,
    sigma_coef=sigma,
    time_min=1e-3,
    time_max=0.5,
    time_schedule='karras',
    return_starting_field=True,
    device=device
)

Random seed set as 42


In [10]:
with torch.no_grad():
    recon_samples = nonlinear_AE.decode(latent_samples.view(-1, 16, 16))
    
latent_samples_mean = latent_samples.mean(dim=1)
recon_samples_mean = recon_samples.view(N, num_ensembles, 64, 64).mean(dim=1)

In [11]:
l_mse = metrics.mse(latent_samples_mean, latent_nonlinear)
l_fro = metrics.frobenius(latent_samples_mean, latent_nonlinear)
p_mse = metrics.mse(recon_samples_mean, test_nonlinear)
p_fro = metrics.frobenius(recon_samples_mean, test_nonlinear)
samples_std = recon_samples.view(N, num_ensembles, 64, 64).std(dim=1)


print(f"Latent MSE: {l_mse:.3e}, Latent Frobenius: {l_fro:.3e}")
print(f"Physical MSE: {p_mse:.3e}, Physical Frobenius: {p_fro:.3e}")
print(f"Sample Standard Deviation: {samples_std.mean():.3e}")

Latent MSE: 4.290e-05, Latent Frobenius: 8.306e-02
Physical MSE: 4.889e-04, Physical Frobenius: 8.985e-02
Sample Standard Deviation: 1.890e-02


Latent Space Flow Matching / Stochastic Interpolants

In [14]:
nonlinear_AE = VariationalAutoEncoder().to(device)
vorticity_AE = VariationalAutoEncoder().to(device)

NONLINEAR_AE_PATH = ROOT / "Trained_Models" / "AE" / "Nonlinear" / "Joint_AE_Nonlinear_SI_GaussianSDE.pth"
VORTICITY_AE_PATH = ROOT / "Trained_Models" / "AE" / "Vorticity" / "Joint_AE_Vorticity_SI_GaussianSDE.pth"
nonlinear_AE.load_state_dict(torch.load(NONLINEAR_AE_PATH))
vorticity_AE.load_state_dict(torch.load(VORTICITY_AE_PATH))

with torch.no_grad():
    latent_nonlinear = nonlinear_AE.encode(test_nonlinear)
    latent_vorticity = vorticity_AE.encode(test_vorticity)
    
    recon_nonlinear = nonlinear_AE.decode(latent_nonlinear)
    recon_vorticity = vorticity_AE.decode(latent_vorticity)
    
AE_recon_re_nonlinear = metrics.frobenius(recon_nonlinear, test_nonlinear)
AE_recon_re_vorticity = metrics.frobenius(recon_vorticity, test_vorticity)
AE_recon_mse_nonlinear = metrics.mse(recon_nonlinear, test_nonlinear)
AE_recon_mse_vorticity = metrics.mse(recon_vorticity, test_vorticity)

    
print(f"AE recon relative L2 error (nonlinear): {AE_recon_re_nonlinear:.4e}")
print(f"AE recon relative L2 error (vorticity): {AE_recon_re_vorticity:.4e}")
print(f"AE recon MSE (nonlinear): {AE_recon_mse_nonlinear:.4e}")
print(f"AE recon MSE (vorticity): {AE_recon_mse_vorticity:.4e}")

AE recon relative L2 error (nonlinear): 2.4272e-02
AE recon relative L2 error (vorticity): 7.6465e-03
AE recon MSE (nonlinear): 4.0419e-05
AE recon MSE (vorticity): 5.6145e-05


In [ ]:
set_seed(42)
num_ensembles = 100

modes = 4
width = 20
Latent_SI = FNO2d_Orig(modes, modes, width, padding = 0, embed_dim = 256, length = 1).to(device)

model_name = ROOT / "Trained_Models" / "SI" / "Latent_SI" / "Joint_SI_Gaussian.pth"
Latent_SI.load_state_dict(torch.load(model_name))

latent_starting_fields, latent_samples, latent_distances = sample_closure_field(
    method='SI_gaussian',
    model=Latent_SI,
    omega=latent_vorticity.float(),
    num_ensembles=num_ensembles,
    sample_steps=10,
    sigma_coef=sigma,
    time_min=1e-3,
    time_max=1.0,
    time_schedule='uniform',
    return_starting_field=True,
    device=device
)

Random seed set as 42


In [18]:
with torch.no_grad():
    recon_samples = nonlinear_AE.decode(latent_samples.view(-1, 16, 16))
    
latent_samples_mean = latent_samples.mean(dim=1)
recon_samples_mean = recon_samples.view(N, num_ensembles, 64, 64).mean(dim=1)

In [19]:
l_mse = metrics.mse(latent_samples_mean, latent_nonlinear)
l_fro = metrics.frobenius(latent_samples_mean, latent_nonlinear)
p_mse = metrics.mse(recon_samples_mean, test_nonlinear)
p_fro = metrics.frobenius(recon_samples_mean, test_nonlinear)
samples_std = recon_samples.view(N, num_ensembles, 64, 64).std(dim=1)


print(f"Latent MSE: {l_mse:.3e}, Latent Frobenius: {l_fro:.3e}")
print(f"Physical MSE: {p_mse:.3e}, Physical Frobenius: {p_fro:.3e}")
print(f"Sample Standard Deviation: {samples_std.mean():.3e}")

Latent MSE: 2.813e-03, Latent Frobenius: 7.355e-02
Physical MSE: 5.132e-04, Physical Frobenius: 9.221e-02
Sample Standard Deviation: 1.506e-02
